# RVQ-Aware Audio Conditioning Evaluation

This notebook compares the fixed-target C2F baseline, the soft-recovery/self-forcing system, and the fully trained L3+L6 audio cross-attention system. It also evaluates deterministic audio corruptions and disables the trained adapters in the same checkpoint for a causal branch ablation.

The workflow has two levels:

1. Deterministic window metrics for every gap from 1 through 15.
2. Full-clip exports and evaluator FID/diversity, R@K, beat, and seam metrics at gaps 3, 7, and 15.

All rows receive identical validation clips and anchors. Audio-ablation rows change only the inference audio tensor. Training `eval/loss` is not used here.

In [ ]:
from pathlib import Path
import gc
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / 'motion_generation').is_dir():
    PROJECT_ROOT = cwd
elif cwd.name == 'notebooks' and cwd.parent.name == 'motion_generation':
    PROJECT_ROOT = cwd.parents[1]
else:
    raise RuntimeError(f'Run from the repository root or motion_generation/notebooks, not {cwd}')

MOTION_GENERATION_DIR = PROJECT_ROOT / 'motion_generation'
if str(MOTION_GENERATION_DIR) not in sys.path:
    sys.path.insert(0, str(MOTION_GENERATION_DIR))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.train_audio_mask_multipart import (
    discover_names,
    load_sequences,
    read_split_file,
)
from utils.multipart_motion import PART_ORDER
from utils.variable_c2f_evaluation import (
    InfillModelSpec,
    add_gap_bucket,
    compute_beat_table,
    compute_fid_table,
    compute_gap_dynamics_table,
    compute_retrieval_table,
    compute_seam_table,
    evaluate_model_windows,
    export_full_clip_gap,
    load_audio_motion_transformer,
    load_part_codecs,
    paired_bootstrap_window_differences,
    summarize_window_metrics,
)

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('project:', PROJECT_ROOT)
print('device:', DEVICE)

## Configuration

`CANONICALIZE_RAW_ROOT=False` preserves the earlier notebook's evaluator protocol and is required for direct comparison with previously reported numbers. A root-canonicalized rerun is useful, but it must be reported as a separate protocol.

Start with `MAX_EVAL_CLIPS=32` and gaps `[3]` for a smoke test. Restore `None` and `range(1, 16)` for the final run.

In [ ]:
DATA_DIR = PROJECT_ROOT / 'SuSuInterActs' / 'SuSuInterActs'
TOKEN_DIR = DATA_DIR / 'motion_token_data_multipart_512x4'
AUDIO_FPS = 10.0
AUDIO_DIR = DATA_DIR / 'audio_features_hubert_layer9_fps10'
EVAL_SPLIT_FILE = DATA_DIR / 'split' / 'val_file_list.txt'
MOTION_DIR = DATA_DIR / 'motion_data'
WAV_DIR = DATA_DIR / 'wav_data'
MOTION2TEXT_JSON = DATA_DIR / 'text_data' / 'motion2text.json'

CHECKPOINT_ROOT = PROJECT_ROOT / 'checkpoints'
FIXED_TARGETS_NO_SF = CHECKPOINT_ROOT / 'mask_multipart_variable_c2f_fixed_targets_no_sf_gap1_15'
SOFT_RECOVERY_SF05 = CHECKPOINT_ROOT / 'mask_multipart_variable_c2f_soft_recovery_sf05_gap1_15'
CROSSATTN_L3_L6 = CHECKPOINT_ROOT / 'mask_multipart_audio_crossattn_l3_l6_gap1_15'
INCLUDE_AUDIO_ABLATIONS = True
INCLUDE_ADAPTER_DISABLED = True

MODEL_SPECS = [
    InfillModelSpec(
        name='fixed_targets_no_sf',
        checkpoint=FIXED_TARGETS_NO_SF,
        decoder='c2f',
        allowed_gaps=tuple(range(1, 16)),
    ),
    InfillModelSpec(
        name='soft_recovery_sf05',
        checkpoint=SOFT_RECOVERY_SF05,
        decoder='c2f',
        allowed_gaps=tuple(range(1, 16)),
    ),
    InfillModelSpec(
        name='crossattn_l3_l6',
        checkpoint=CROSSATTN_L3_L6,
        decoder='c2f',
        allowed_gaps=tuple(range(1, 16)),
    ),
]
if INCLUDE_AUDIO_ABLATIONS:
    for mode in ('temporal_shift', 'temporal_shuffle', 'cross_clip', 'temporal_mean', 'zero'):
        MODEL_SPECS.append(InfillModelSpec(
            name=f'crossattn_l3_l6_{mode}',
            checkpoint=CROSSATTN_L3_L6,
            decoder='c2f',
            allowed_gaps=tuple(range(1, 16)),
            audio_input_mode=mode,
            audio_ablation_seed=42,
        ))
if INCLUDE_ADAPTER_DISABLED:
    MODEL_SPECS.append(InfillModelSpec(
        name='crossattn_l3_l6_adapters_disabled',
        checkpoint=CROSSATTN_L3_L6,
        decoder='c2f',
        allowed_gaps=tuple(range(1, 16)),
        disable_audio_adapters=True,
    ))

RVQ_ROOT = CHECKPOINT_ROOT / 'multipart_rvqvae'
PART_CHECKPOINTS = {
    part: RVQ_ROOT / f'rvq_{part}_512x4_bs256_cosine' / 'model' / 'best.pth'
    for part in PART_ORDER
}

WINDOW_GAPS = list(range(1, 16))
FULL_CLIP_GAPS = [3, 7, 15]
WINDOWS_PER_CLIP_PER_GAP = 1
WINDOW_BATCH_SIZE = 32
EXPORT_BATCH_SIZE = 32
EVAL_SEED = 42
MAX_EVAL_CLIPS = None
SLOT_CONSTRAINED = False  # Keep False for the primary trained-decoder result.

GT_SOURCE = 'raw_motion_data'
CANONICALIZE_RAW_ROOT = False
OUT_ROOT = MOTION_GENERATION_DIR / 'notebooks' / 'audio_crossattn_full_eval_outputs'
WINDOW_CSV = OUT_ROOT / 'window_metrics.csv'

EVALUATION_DIR = PROJECT_ROOT / 'evaluation'
EVALUATOR_CKPT = CHECKPOINT_ROOT / 'eval_model' / 'best_model.pt'
EVALUATOR_CONFIG = EVALUATION_DIR / 'config' / 'train_bert_orig.yaml'
EVALUATOR_STATS_DIR = EVALUATION_DIR / 'stats' / 'humanml3d' / 'guoh3dfeats'
EVALUATOR_TEXT_MODEL = os.environ.get('EVALUATOR_TEXT_MODEL_PATH', 'bert-base-chinese')

RUN_WINDOW_EVAL = True
RUN_FULL_CLIP_EXPORT = False
RUN_FID_DIVERSITY = False
RUN_RETRIEVAL_RK = False
RUN_SEAM_METRICS = False
RUN_BEAT_METRICS = False
RUN_DYNAMICS_METRICS = False
BOOTSTRAP_ITERATIONS = 2000

In [ ]:
checks = []
for spec in MODEL_SPECS:
    checks.append({
        'name': spec.name,
        'path': str(spec.checkpoint),
        'directory': spec.checkpoint.is_dir(),
        'config': (spec.checkpoint / 'config.json').exists(),
        'weights': (spec.checkpoint / 'model.safetensors').exists() or (spec.checkpoint / 'pytorch_model.bin').exists(),
        'decoder': spec.decoder,
        'audio_input': spec.audio_input_mode,
        'adapters_disabled': spec.disable_audio_adapters,
        'gaps': f'{min(spec.allowed_gaps)}-{max(spec.allowed_gaps)}',
    })
display(pd.DataFrame(checks).set_index('name'))
for part, path in PART_CHECKPOINTS.items():
    print(f'{part:6s}', path, 'exists=', path.exists())
print('tokens:', TOKEN_DIR, TOKEN_DIR.exists())
print('audio:', AUDIO_DIR, AUDIO_DIR.exists())
print('split:', EVAL_SPLIT_FILE, EVAL_SPLIT_FILE.exists())

## Load Codecs and Validation Data

The codec checkpoints are shared by every model. The same loaded sequence objects are reused so window selection cannot drift between architectures or audio conditions.

In [ ]:
codecs = load_part_codecs(PART_CHECKPOINTS, DEVICE)
codebook_size = next(iter(codecs.values())).codebook_size
num_quantizers = next(iter(codecs.values())).num_quantizers
tokens_per_frame = len(PART_ORDER) * num_quantizers
codec_unit_length = next(iter(codecs.values())).unit_length

eval_names = discover_names(TOKEN_DIR, AUDIO_DIR, read_split_file(EVAL_SPLIT_FILE))
eval_sequences, load_stats = load_sequences(
    eval_names,
    TOKEN_DIR,
    AUDIO_DIR,
    codebook_size=codebook_size,
    num_tokens_per_frame=tokens_per_frame,
    audio_fps=AUDIO_FPS,
    source_motion_fps_fallback=20.0,
    motion_token_fps_override=None,
    motion_token_unit_length_override=None,
    max_sequences=MAX_EVAL_CLIPS,
)

print('part order:', PART_ORDER)
print('codebook:', codebook_size, 'quantizers:', num_quantizers, 'tokens/frame:', tokens_per_frame)
print('codec unit length:', codec_unit_length)
print('validation sequences:', len(eval_sequences), load_stats)

## Deterministic Window Evaluation

Every enabled C2F row is evaluated on the same deterministic windows. Token metrics use canonical codec IDs, and decoded metrics measure the resulting motion. Audio controls are deterministic and do not modify checkpoint weights.

In [ ]:
if RUN_WINDOW_EVAL:
    window_frames = []
    parameter_rows = []
    for spec in MODEL_SPECS:
        print(f'Loading {spec.name}: {spec.checkpoint}')
        model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
        parameter_rows.append({
            'model': spec.name,
            'checkpoint': str(spec.checkpoint),
            'audio_input': spec.audio_input_mode,
            'adapters_disabled': spec.disable_audio_adapters,
            'fusion_mode': model.config.audio_fusion_mode,
            'adapter_layers': tuple(model.config.audio_adapter_layers),
            'total_parameters': sum(p.numel() for p in model.parameters()),
        })
        frame = evaluate_model_windows(
            model,
            spec,
            eval_sequences,
            codecs,
            WINDOW_GAPS,
            batch_size=WINDOW_BATCH_SIZE,
            device=DEVICE,
            windows_per_clip=WINDOWS_PER_CLIP_PER_GAP,
            seed=EVAL_SEED,
            slot_constrained=SLOT_CONSTRAINED,
        )
        window_frames.append(frame)
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    window_df = add_gap_bucket(pd.concat(window_frames, ignore_index=True))
    OUT_ROOT.mkdir(parents=True, exist_ok=True)
    window_df.to_csv(WINDOW_CSV, index=False)
    parameter_df = pd.DataFrame(parameter_rows).drop_duplicates('checkpoint')
    parameter_df.to_csv(OUT_ROOT / 'parameter_counts.csv', index=False)
else:
    window_df = pd.read_csv(WINDOW_CSV)
    window_df = add_gap_bucket(window_df)

print('window rows:', len(window_df))
display(window_df.head())
if RUN_WINDOW_EVAL:
    display(parameter_df)

In [ ]:
by_gap, macro = summarize_window_metrics(window_df)
display(by_gap.round(5))
print('Macro average across gaps:')
display(macro.round(5))

gap3_columns = [
    'token_acc', 'q0_acc', 'q1_acc', 'q2_acc', 'q3_acc',
    'invalid_token_frac', 'body_rmse', 'body_velocity_rmse',
]
gap3 = window_df[window_df['gap'] == 3].groupby('model')[gap3_columns].mean()
print('Controlled gap-3 comparison:')
display(gap3.round(5))

bucket_metrics = ['token_acc', 'q0_acc', 'body_rmse', 'body_velocity_rmse', 'invalid_token_frac']
bucket_summary = (
    window_df.groupby(['model', 'gap_bucket'], observed=True)[bucket_metrics]
    .mean()
)
display(bucket_summary.round(5))

audio_rows = ['crossattn_l3_l6'] + [
    f'crossattn_l3_l6_{mode}'
    for mode in ('temporal_shift', 'temporal_shuffle', 'cross_clip', 'temporal_mean', 'zero')
] + ['crossattn_l3_l6_adapters_disabled']
available = [name for name in audio_rows if name in macro.index]
if len(available) > 1:
    print('Same-checkpoint audio dependence:')
    display(macro.loc[available].round(5))

## Paired Window Bootstrap

Differences are paired by clip, left anchor, and gap, then bootstrapped over clips. The system table compares both trained systems with the fixed-target starting checkpoint. The causal table compares adapter-on against the same full checkpoint with its cross-attention residuals disabled. Positive accuracy differences are favorable; negative RMSE differences are favorable.

In [ ]:
bootstrap_metrics = [
    'token_acc', 'q0_acc', 'body_rmse',
    'body_velocity_rmse', 'body_acceleration_rmse',
]
system_bootstrap = paired_bootstrap_window_differences(
    window_df,
    reference_model='fixed_targets_no_sf',
    candidate_models=['soft_recovery_sf05', 'crossattn_l3_l6'],
    metrics=bootstrap_metrics,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=EVAL_SEED,
)
adapter_bootstrap = paired_bootstrap_window_differences(
    window_df,
    reference_model='crossattn_l3_l6_adapters_disabled',
    candidate_models=['crossattn_l3_l6'],
    metrics=bootstrap_metrics,
    iterations=BOOTSTRAP_ITERATIONS,
    seed=EVAL_SEED,
)
system_bootstrap.to_csv(OUT_ROOT / 'paired_system_bootstrap.csv', index=False)
adapter_bootstrap.to_csv(OUT_ROOT / 'paired_adapter_bootstrap.csv', index=False)
print('System comparison against fixed-target baseline:')
display(system_bootstrap.round(6))
print('Causal adapter-on minus adapter-disabled comparison:')
display(adapter_bootstrap.round(6))

In [ ]:
plot_frame = by_gap.reset_index()
fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True)
for model_name, group in plot_frame.groupby('model'):
    axes[0, 0].plot(group['gap'], group['token_acc'], marker='o', label=model_name)
    axes[0, 1].plot(group['gap'], group['q0_acc'], marker='o', label=model_name)
    axes[1, 0].plot(group['gap'], group['body_rmse'], marker='o', label=model_name)
    axes[1, 1].plot(group['gap'], group['body_velocity_rmse'], marker='o', label=model_name)
axes[0, 0].set_title('Original-token accuracy')
axes[0, 1].set_title('Coarse q0 accuracy')
axes[1, 0].set_title('Decoded body RMSE')
axes[1, 1].set_title('Decoded velocity RMSE')
for axis in axes.flat:
    axis.set_xlabel('Gap token frames')
    axis.grid(alpha=0.25)
axes[0, 0].set_ylabel('Higher is better')
axes[0, 1].set_ylabel('Higher is better')
axes[1, 0].set_ylabel('Lower is better')
axes[1, 1].set_ylabel('Lower is better')
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3)
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

## Optional Slot-Constrained Decoding Ablation

The primary result above evaluates the decoder implemented during training. A slot-constrained C2F pass prevents a token position from selecting another part or quantizer's vocabulary range. Run this only as a separately named decoding ablation; do not overwrite the primary result.

In [ ]:
RUN_SLOT_CONSTRAINED_ABLATION = False
if RUN_SLOT_CONSTRAINED_ABLATION:
    ablation_frames = []
    for original_spec in MODEL_SPECS[:3]:
        spec = InfillModelSpec(
            name=original_spec.name + '_slot_constrained',
            checkpoint=original_spec.checkpoint,
            decoder=original_spec.decoder,
            allowed_gaps=original_spec.allowed_gaps,
            audio_input_mode=original_spec.audio_input_mode,
        )
        model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
        ablation_frames.append(evaluate_model_windows(
            model, spec, eval_sequences, codecs, WINDOW_GAPS,
            batch_size=WINDOW_BATCH_SIZE, device=DEVICE,
            windows_per_clip=WINDOWS_PER_CLIP_PER_GAP, seed=EVAL_SEED,
            slot_constrained=True,
        ))
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    slot_ablation_df = add_gap_bucket(pd.concat(ablation_frames, ignore_index=True))
    slot_ablation_df.to_csv(OUT_ROOT / 'window_metrics_slot_constrained.csv', index=False)
    display(summarize_window_metrics(slot_ablation_df)[1].round(5))

## Full-Clip Export

Set `RUN_FULL_CLIP_EXPORT=True` after the window smoke test. Every architecture and audio-control row is exported at gaps 3, 7, and 15 with the same `gt` and `mp_codec_gt` references.

In [ ]:
if RUN_FULL_CLIP_EXPORT:
    manifests = []
    for gap in FULL_CLIP_GAPS:
        for spec in MODEL_SPECS:
            if not spec.supports_gap(gap):
                continue
            model = load_audio_motion_transformer(spec.checkpoint, DEVICE)
            manifest = export_full_clip_gap(
                model, spec, eval_sequences, codecs,
                gap=gap, batch_size=EXPORT_BATCH_SIZE, device=DEVICE,
                output_root=OUT_ROOT / 'full_clips', motion_dir=MOTION_DIR,
                gt_source=GT_SOURCE, canonicalize_raw_root=CANONICALIZE_RAW_ROOT,
                slot_constrained=SLOT_CONSTRAINED, clean=True,
            )
            manifests.append(manifest)
            del model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    full_manifest = pd.concat(manifests, ignore_index=True)
    full_manifest.to_csv(OUT_ROOT / 'full_clip_manifest.csv', index=False)
    display(full_manifest.groupby(['model', 'gap']).size().rename('clips'))
else:
    print('Set RUN_FULL_CLIP_EXPORT=True, run this cell, then enable the metric cells below.')

## Evaluator FID and Diversity

Compare the four correct-audio architecture rows first. The same-checkpoint corruption rows then test whether the selected model actually depends on aligned audio.

In [ ]:
if RUN_FID_DIVERSITY:
    fid_frames = []
    for gap in FULL_CLIP_GAPS:
        names = [spec.name for spec in MODEL_SPECS if spec.supports_gap(gap)]
        table = compute_fid_table(
            OUT_ROOT / 'full_clips' / f'gap{gap:02d}', names,
            evaluation_dir=EVALUATION_DIR, evaluator_checkpoint=EVALUATOR_CKPT,
            evaluator_config=EVALUATOR_CONFIG, evaluator_stats_dir=EVALUATOR_STATS_DIR,
            device=DEVICE, batch_size=64, diversity_times=300,
        ).reset_index()
        table['gap'] = gap
        fid_frames.append(table)
    fid_df = pd.concat(fid_frames, ignore_index=True).set_index(['gap', 'model'])
    fid_df.to_csv(OUT_ROOT / 'fid_diversity_metrics.csv')
    display(fid_df.round(6))
    fid_plot = fid_df.reset_index()
    fig, axis = plt.subplots(figsize=(12, 5))
    for model_name, group in fid_plot.groupby('model'):
        group = group.sort_values('gap')
        axis.plot(group['gap'], group['FID_norm_by_GT'], marker='o', label=model_name)
    axis.set(title='FID normalized by GT', xlabel='Gap token frames', ylabel='Lower is better')
    axis.grid(alpha=0.25)
    axis.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    plt.show()
else:
    print('Set RUN_FID_DIVERSITY=True after full-clip export.')

## R@K Semantic Retrieval

This requires a local `bert-base-chinese` directory when the machine is offline. Set `EVALUATOR_TEXT_MODEL_PATH` before starting Jupyter or replace `EVALUATOR_TEXT_MODEL` above.

In [ ]:
if RUN_RETRIEVAL_RK:
    retrieval_frames = []
    for gap in FULL_CLIP_GAPS:
        names = [spec.name for spec in MODEL_SPECS if spec.supports_gap(gap)]
        table = compute_retrieval_table(
            OUT_ROOT / 'full_clips' / f'gap{gap:02d}', names,
            evaluation_dir=EVALUATION_DIR, evaluator_checkpoint=EVALUATOR_CKPT,
            evaluator_config=EVALUATOR_CONFIG, evaluator_stats_dir=EVALUATOR_STATS_DIR,
            motion2text_json=MOTION2TEXT_JSON, text_model_path=EVALUATOR_TEXT_MODEL,
            device=DEVICE,
        ).reset_index()
        table['gap'] = gap
        retrieval_frames.append(table)
    retrieval_df = pd.concat(retrieval_frames, ignore_index=True).set_index(['gap', 'model'])
    retrieval_df.to_csv(OUT_ROOT / 'retrieval_rk_metrics.csv')
    display(retrieval_df.round(4))
    retrieval_plot = retrieval_df.reset_index()
    fig, axis = plt.subplots(figsize=(12, 5))
    for model_name, group in retrieval_plot.groupby('model'):
        group = group.sort_values('gap')
        axis.plot(group['gap'], group['t2m/R01'], marker='o', label=model_name)
    axis.set(title='Text-to-motion R@1', xlabel='Gap token frames', ylabel='Higher is better')
    axis.grid(alpha=0.25)
    axis.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    plt.show()
else:
    print('Set RUN_RETRIEVAL_RK=True after full-clip export.')

## Seam and Beat Metrics

Seam boundaries are derived from `(gap + 1) * codec_unit_length`, so each gap is measured at its actual anchor stride. In addition to raw discontinuity, seam excess is normalized by ordinary non-seam dynamics. Beat diagnostics include event counts, both ESD directions, and velocity-onset correlation.

In [ ]:
if RUN_SEAM_METRICS:
    seam_frames = []
    for gap in FULL_CLIP_GAPS:
        names = [spec.name for spec in MODEL_SPECS if spec.supports_gap(gap)]
        table = compute_seam_table(
            OUT_ROOT / 'full_clips' / f'gap{gap:02d}', names,
            gap=gap, codec_unit_length=codec_unit_length,
        ).reset_index()
        table['gap'] = gap
        seam_frames.append(table)
    seam_df = pd.concat(seam_frames, ignore_index=True).set_index(['gap', 'model'])
    seam_df.to_csv(OUT_ROOT / 'seam_metrics.csv')
    display(seam_df.round(6))

if RUN_BEAT_METRICS:
    beat_frames = []
    for gap in FULL_CLIP_GAPS:
        names = [spec.name for spec in MODEL_SPECS if spec.supports_gap(gap)]
        table = compute_beat_table(
            OUT_ROOT / 'full_clips' / f'gap{gap:02d}', names,
            evaluation_dir=EVALUATION_DIR, wav_dir=WAV_DIR, fps=20, tolerance=0.1,
        ).reset_index()
        table['gap'] = gap
        beat_frames.append(table)
    beat_df = pd.concat(beat_frames, ignore_index=True).set_index(['gap', 'model'])
    beat_df.to_csv(OUT_ROOT / 'beat_metrics.csv')
    display(beat_df.round(6))

if RUN_DYNAMICS_METRICS:
    dynamics_frames = []
    for gap in FULL_CLIP_GAPS:
        names = [spec.name for spec in MODEL_SPECS if spec.supports_gap(gap)]
        table = compute_gap_dynamics_table(
            OUT_ROOT / 'full_clips' / f'gap{gap:02d}', names,
            gap=gap, codec_unit_length=codec_unit_length, feature_start=3,
        ).reset_index()
        table['gap'] = gap
        dynamics_frames.append(table)
    dynamics_df = pd.concat(dynamics_frames, ignore_index=True).set_index(['gap', 'model'])
    dynamics_df.to_csv(OUT_ROOT / 'gap_dynamics_metrics.csv')
    display(dynamics_df.round(6))

if not RUN_SEAM_METRICS and not RUN_BEAT_METRICS and not RUN_DYNAMICS_METRICS:
    print('Enable seam, beat, and/or dynamics diagnostics after full-clip export.')

## Reporting Rules

Use `fixed_targets_no_sf` as the starting-checkpoint reference. Treat `soft_recovery_sf05` versus `crossattn_l3_l6` as an end-to-end system comparison because their recovery objectives, self-forcing behavior, and audio architecture differ.

Use `crossattn_l3_l6` versus `crossattn_l3_l6_adapters_disabled` as the causal adapter comparison. The disabled row preserves the trained routed-additive path, part/RVQ/stage embeddings, and all base weights. Audio-corruption rows test dependence on correct audio but do not isolate cross-attention. Treat raw seam magnitude as a smoothness-sensitive quantity: report seam excess, generated-gap dynamic ratios to codec GT, motion/audio event counts, directional ESD, and VOC alongside it. Report `mp_codec_gt` with every full-clip table, and keep slot-constrained or root-canonicalized runs as separate protocols.